# 🧠 Transfer Learning
**Course:** Deep Learning  
**Lecture:** 3 - Convolutional Neural Networks (CNNs)  
**Part:** d - Transfer Learning

---

> **Where we are:** Training deep CNNs from scratch takes days and millions of images. We don't have that. Transfer learning lets us reuse knowledge from pre-trained models.

**What you'll learn:**
- Feature extraction vs Fine-tuning
- How to freeze and unfreeze layers in Keras
- Building custom classification heads

**Exam relevance:** ⭐⭐⭐ (This is how deep learning is done in the real world.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 Theory: Transfer Learning Strategies
Transfer learning takes a model trained on a large dataset (like ImageNet) and adapts it to a new, smaller dataset.

1. **Feature Extraction:** Freeze the entire pre-trained base model. Add a new head and train ONLY the head. Use this when the new dataset is small and similar to the original.
2. **Fine-Tuning:** Unfreeze the top layers of the base model. Train the head AND the top layers with a very small learning rate. Use this when you have more data or the dataset is slightly different.

## 🔀 Your Options
At this step, you can choose between:
| Option | Pros | Cons | When to use |
|---|---|---|---|
| From Scratch | Custom architecture | Needs huge data, prone to overfit | Novel problems/data types |
| Feature Extraction | Fast, low risk of overfitting | Limited adaptation capability | Small datasets |
| Fine-Tuning | Higher accuracy, adapts features | Can destroy pre-trained weights if LR is too high | Medium/large datasets |
| Full Fine-Tuning | Maximum adaptability | Extremely high risk of overfitting | Huge datasets |

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train, x_test = x_train[:2000], x_test[:500]
y_train, y_test = y_train[:2000], y_test[:500]
x_train = tf.image.resize(x_train, (96, 96)) / 127.5 - 1.0
x_test  = tf.image.resize(x_test,  (96, 96)) / 127.5 - 1.0
y_train_cat = tf.keras.utils.to_categorical(y_train, 10)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  10)

def build_feature_extractor():
    # Load MobileNetV2 without classification head, pretrained on ImageNet
    base = tf.keras.applications.MobileNetV2(
        input_shape=(96, 96, 3), include_top=False, weights='imagenet'
    )
    base.trainable = False  # freeze ALL base layers
    inp = tf.keras.Input(shape=(96, 96, 3))
    x = base(inp, training=False)            # inference mode even during training
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(10, activation='softmax')(x)
    model = Model(inp, out, name='FeatureExtractor')
    return model, base

def build_finetuned(base, fine_tune_at=100):
    # Unfreeze layers after index fine_tune_at
    base.trainable = True
    for layer in base.layers[:fine_tune_at]:
        layer.trainable = False
    inp = tf.keras.Input(shape=(96, 96, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(10, activation='softmax')(x)
    model = Model(inp, out, name='FineTuned')
    return model

# --- Strategy 1: Feature extraction (frozen base) ---
model_fe, base_model = build_feature_extractor()
model_fe.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
trainable = sum(tf.size(v).numpy() for v in model_fe.trainable_variables)
frozen   = sum(tf.size(v).numpy() for v in model_fe.non_trainable_variables)
print(f'Feature extractor: {trainable:,} trainable, {frozen:,} frozen params')
history_fe = model_fe.fit(x_train, y_train_cat, epochs=3, validation_data=(x_test, y_test_cat), verbose=1)

# --- Strategy 2: Fine-tuning (unfreeze top layers, lower lr) ---
model_ft = build_finetuned(base_model, fine_tune_at=100)
model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_ft = model_ft.fit(x_train, y_train_cat, epochs=3, validation_data=(x_test, y_test_cat), verbose=1)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_fe.history[f'val_{metric}'], label='Feature extraction', marker='o')
    ax.plot(history_ft.history[f'val_{metric}'], label='Fine-tuning', marker='s')
    ax.set_title(f'Validation {title}'); ax.set_xlabel('Epoch'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ✅ Solution
def build_feature_extractor():
    base_model = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
    
    # FREEZE
    base_model.trainable = False 
    
    inputs = Input(shape=(96, 96, 3))
    x = base_model(inputs, training=False) # training=False crucial for BatchNorm layers!
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(10, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    return model, base_model

model_ext, base_model = build_feature_extractor()

In [ ]:
# 🔬 Comparison Cell: Feature Extraction vs Fine Tuning
import pandas as pd
stats = []

print("--- Phase 1: Feature Extraction ---")
model_ext.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
trainable_params_1 = sum([tf.keras.backend.count_params(w) for w in model_ext.trainable_weights])
frozen_params_1 = sum([tf.keras.backend.count_params(w) for w in model_ext.non_trainable_weights])

h1 = model_ext.fit(x_train, y_train, epochs=3, validation_split=0.2, verbose=0)
stats.append({'Strategy': 'Feature Extraction', 'Trainable Params': trainable_params_1, 'Frozen Params': frozen_params_1, 'Val Accuracy': h1.history['val_accuracy'][-1]})

print("\n--- Phase 2: Fine Tuning ---")
# Unfreeze the base model top layers
base_model.trainable = True
# Freeze early layers (MobileNetV2 has ~154 layers)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with very small learning rate!
model_ext.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
trainable_params_2 = sum([tf.keras.backend.count_params(w) for w in model_ext.trainable_weights])
frozen_params_2 = sum([tf.keras.backend.count_params(w) for w in model_ext.non_trainable_weights])

h2 = model_ext.fit(x_train, y_train, epochs=3, validation_split=0.2, verbose=0)
stats.append({'Strategy': 'Fine-Tuning', 'Trainable Params': trainable_params_2, 'Frozen Params': frozen_params_2, 'Val Accuracy': h2.history['val_accuracy'][-1]})

display(pd.DataFrame(stats))

# Plotting
acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.plot(acc, marker='o')
plt.axvline(x=2.5, color='r', linestyle='--', label='Start Fine Tuning')
plt.title('Validation Accuracy: Extractor -> Fine Tuning')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 🎓 Exam Corner

### Question 1 (Conceptual)
Why do we freeze the base model layers during the feature extraction phase?

<details><summary>💡 Answer</summary>

To prevent the randomly initialized weights of our new classification head from sending massive, chaotic error gradients backward during the first few epochs. This would destroy the carefully learned weights of the pre-trained model.

</details>

### Question 2 (Conceptual)
Why should you use a smaller learning rate when fine-tuning?

<details><summary>💡 Answer</summary>

Because the pre-trained weights are already quite good. We only want to make small, incremental adjustments to adapt them to the new dataset. A large learning rate would cause catastrophic forgetting of the pre-trained knowledge.

</details>

### Question 3 (Conceptual)
What is the key risk of full fine-tuning when you have very little target data?

<details><summary>💡 Answer</summary>

Overfitting. A deep model like VGG or MobileNet has millions of parameters. With very little target data and full fine-tuning, the model will simply memorize the small dataset and fail to generalize.

</details>